In [ ]:
%pip -q install -U asf_search numpy pandas matplotlib geopandas shapely h5py pyproj rasterio scipy tqdm


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import asf_search as asf
from shapely.geometry import box, shape
from IPython.display import display

# Просторові рамки — лише для пошуку/відображення метаданих
UKRAINE_BBOX = (22.0, 44.0, 40.3, 52.5)
# Широкий район колишнього водосховища; для аналізу використовується окремий полігон нижче
KAKHOVKA_BBOX = (32.35, 46.32, 35.55, 47.55)
START, END = '2026-10-21', '2026-11-16'  # кінцева дата не включається в деяких API
MAX_RESULTS = 250
DATA_DIR = Path('..') / 'data' / 'h5'; DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def find_gcov(bbox, start=START, end=END, max_results=MAX_RESULTS, maturity='VALIDATED'):
    geom = box(*bbox).wkt
    return asf.search(dataset='NISAR', processingLevel='GCOV', dataMaturity=maturity,
                      intersectsWith=geom, start=start, end=end, maxResults=max_results)

def result_table(results):
    rows=[]
    for i, r in enumerate(results):
        p=r.properties
        rows.append({'i':i,'start':p.get('startTime'),'stop':p.get('stopTime'),
                     'flightDirection':p.get('flightDirection'),
                     'path':p.get('pathNumber'),'frame':p.get('frameNumber'),
                     'size_MB': round(sum(v.get('bytes', 0) for v in p['bytes'].values()) / 1e6, 1) if isinstance(p.get('bytes'), dict) else (round(p['bytes'] / 1e6, 1) if p.get('bytes') else None),
                     'scene':p.get('sceneName') or p.get('fileID') or str(r)[:95]})
    return pd.DataFrame(rows)

ukr = find_gcov(UKRAINE_BBOX)
print('Покриття України (необов’язково повний перелік; maxResults =',MAX_RESULTS,'):', len(ukr))
ukr_table=result_table(ukr)
if not ukr_table.empty:
    print('Кількість сцен за днями:')
    display(ukr_table.assign(day=pd.to_datetime(ukr_table.start).dt.date).groupby('day').size().rename('scenes').to_frame())

kh = find_gcov(KAKHOVKA_BBOX)
kh_table = result_table(kh)
print('Сцен над районом Каховського водосховища:', len(kh))
display(kh_table.head(40))
if kh_table.empty:
    print('Немає результатів у цьому інтервалі: спробуйте START=2025-11-01 і END=2026-11-16, maturity=PROVISIONAL або BETA.')